# Imports

In [ ]:
from transformers import ViTMAEForPreTraining
from transformers import AutoImageProcessor
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import os
from PIL import Image

# Checking Device

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")   # Apple GPU
    print("Using MPS (Apple GPU)")
elif torch.cuda.is_available():
    device = torch.device("cuda")  # NVIDIA GPU
    print("Using CUDA")
else:
    device = torch.device("cpu")   # fallback
    print("Using CPU")

# Importing Vision Transformer Backbone

In [ ]:
model = ViTMAEForPreTraining.from_pretrained("facebook/vit-mae-base")
model.to(device)

# Setting up Image Preprocessor

In [ ]:
processor = AutoImageProcessor.from_pretrained("facebook/vit-mae-base")

# Defining FIDD Dataset

In [ ]:
class FIDDDataset(Dataset):
    def __init__(self, root_dir, processor, split='train'):
        # This points to /home/le.song/FIDD_Data/Stable_Subset/Real/train (or val)
        self.target_dir = os.path.join(root_dir, split)
        self.processor = processor
        self.image_paths = []

        if not os.path.exists(self.target_dir):
            raise FileNotFoundError(f"Directory not found: {self.target_dir}")

        # Gather all images directly from the split folder
        valid_extensions = (".png", ".jpg", ".jpeg")
        for f in os.listdir(self.target_dir):
            if f.lower().endswith(valid_extensions):
                self.image_paths.append(os.path.join(self.target_dir, f))
        
        print(f"--- {split.capitalize()} Dataset Initialized ---")
        print(f"Found {len(self.image_paths)} images in {self.target_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = Image.open(img_path).convert("RGB")
        
        # ViTMAE expects images normalized / resized via processor
        pixel_values = self.processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0)
        return pixel_values

# Data Loaders

In [ ]:
train_dataset = FIDDDataset(root_dir="/home/le.song/FIDD_Data/Stable_Subset/Real", processor=processor, split='train')
val_dataset = FIDDDataset(root_dir="/home/le.song/FIDD_Data/Stable_Subset/Real", processor=processor, split='val')

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

# Optimizer

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
num_epochs = 1  # start small to verify

# Early Stopper with Patience 3

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

# Training Loop

In [ ]:
import torch

def train_mae(model, train_loader, val_loader, device, num_epochs=20, lr=1e-4, save_path="/home/le.song/mae_fidd_model"):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    
    # Scheduler: Reduce LR by factor of 0.1 if val_loss doesn't improve for 2 epochs
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2, verbose=True)
    
    # Early Stopper: Stop training if no improvement for 3 epochs
    early_stopper = EarlyStopping(patience=3)
    early_stopper.best_loss = None 

    os.makedirs(save_path, exist_ok=True)
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for batch in train_loader:
            batch = batch.to(device)
            outputs = model(pixel_values=batch)
            loss = outputs.loss
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for val_batch in val_loader:
                val_batch = val_batch.to(device)
                val_outputs = model(pixel_values=val_batch)
                val_loss += val_outputs.loss.item()
        
        avg_val_loss = val_loss / len(val_loader)
        
        print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

        # Step Scheduler
        scheduler.step(avg_val_loss)

        # Save Best Model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            model.save_pretrained(save_path)
            processor.save_pretrained(save_path)
            print(f"--> Saved new best model (Val Loss: {best_val_loss:.4f})")

        # Check Early Stopping
        early_stopper(avg_val_loss)
        if early_stopper.early_stop:
            print("Early stopping triggered. Training halted.")
            break

    return model

# Training

In [ ]:
model = train_mae(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    num_epochs=20,
    lr=1e-4
)

# Visualize Reconstructions

In [ ]:
"""# Visualize Reconstructions"""
import matplotlib.pyplot as plt

def unpatchify(x, patch_size=16):
    p = patch_size
    h = w = int(x.shape[1]**.5)
    x = x.reshape(shape=(x.shape[0], h, w, p, p, 3))
    x = torch.einsum('nhwpqc->nchpwq', x)
    imgs = x.reshape(shape=(x.shape[0], 3, h * p, h * p))
    return imgs

model.eval()
batch = next(iter(val_loader)).to(device)

with torch.no_grad():
    outputs = model(pixel_values=batch)
    # Use .logits and reshape back to image dimensions
    reconstructed = unpatchify(outputs.logits)

# First 5 images
num_show = 5
fig, axes = plt.subplots(num_show, 2, figsize=(8, num_show*4))

# Standard ImageNet values for denormalization
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

for i in range(num_show):
    # Denormalize and clip for display
    orig = torch.clip((batch[i].cpu() * std) + mean, 0, 1).permute(1, 2, 0)
    recon = torch.clip((reconstructed[i].cpu() * std) + mean, 0, 1).permute(1, 2, 0)

    axes[i, 0].imshow(orig)
    axes[i, 0].set_title("Original")
    axes[i, 0].axis("off")
    
    axes[i, 1].imshow(recon)
    axes[i, 1].set_title("Reconstructed (Masked)")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.savefig("final_reconstruction.png")
plt.show()